# 03 - Modelagem Supervisionada de Atrasos

Modelos de classifica??o para prever atraso >15 minutos.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

data = pd.read_parquet("../data/processed/flights_sample.parquet")
print(data.shape)

## 1. Sele??o de features e alvo
Usamos apenas informa??es dispon?veis antes da decolagem para evitar vazamento.

In [ ]:
features = ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT","MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE","IS_WEEKEND"]
X = data[features]
y = data["DELAYED"]

cat_features = ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT"]
num_features = ["MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE","IS_WEEKEND"]

## 2. Divis?o treino/teste estratificada

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

## 3. Pipeline de pr?-processamento
- Imputa??o para robustez.
- One-hot para categ?ricas.
- Pass-through para num?ricas.

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, cat_features),
        ("num", numerical_transformer, num_features)
    ]
)

## 4. Baseline: Regress?o Log?stica (subamostra)

In [ ]:
log_reg = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=300, n_jobs=-1))
])

X_train_lr = X_train.sample(n=100_000, random_state=42)
y_train_lr = y_train.loc[X_train_lr.index]

log_reg.fit(X_train_lr, y_train_lr)
y_pred_lr = log_reg.predict(X_test)
y_proba_lr = log_reg.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

## 5. Modelo principal: Random Forest

In [ ]:
rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=120, max_depth=16, random_state=42, n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

## 6. Matriz de confus?o (Random Forest)

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=["No atraso","Atraso"], yticklabels=["No atraso","Atraso"])
plt.title("Matriz de confus?o normalizada")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()

### Conclus?es
- Random Forest supera o baseline em ROC-AUC.
- Features temporais e de rota trazem maior sinal.
- Fluxo pronto para exportar modelo treinado.